# 4 — Contrast QC

**Runs on CPU in seconds. No model weights, no data download.**

This is the shortest tutorial here and the one most likely to save you.

Every panel in [Tutorials 2](02_subtype_markers) and [3](03_choosing_the_reference) came from a
contrast that was real: the target and the reference genuinely occupied different regions of the
encoder's space, so the direction `u = mean(Z_target) − mean(Z_reference)` was pinned by that
separation.

When they *don't* separate, `u` is estimated from sampling noise and points somewhere arbitrary.
The critical property of that failure is that **it has no symptom in the output.** You still get a
ranked list. It is still the right length. The gene names still look like gene names.

So let us build that failure on purpose and look at what it hands back.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np, pandas as pd, anndata as ad
import recast
from recast.encoders import StubEncoder
from recast.qc import ContrastQCWarning

warnings.simplefilter("always", ContrastQCWarning)   # see every warning, not just the first

NC, NG = 400, 200
rng = np.random.default_rng(0)
X = rng.poisson(8, (NC, NG)).astype("float32")       # ONE homogeneous population
genes = [f"G{i:03d}" for i in range(NG)]
print("one population:", X.shape, "— no subgroup structure of any kind")

one population: (400, 200) — no subgroup structure of any kind


## A cluster boundary that isn't there

There is exactly one population above. We now split it into "cluster A" and "cluster B"
**at random** — the kind of boundary an over-clustered resolution parameter invents — and ask
RECAST what makes A different from B.

The honest answer is *nothing*. Here is the answer it actually gives, twice, under two
independent random splits of the same cells.

In [2]:
def panel_from_random_split(seed, k=10):
    A = ad.AnnData(X.copy()); A.var_names = genes
    lab = np.array(["B"] * NC)
    lab[np.random.default_rng(seed).permutation(NC)[:NC // 2]] = "A"
    A.obs["cluster"] = pd.Categorical(lab)
    res = recast.attribute(StubEncoder(NG), A, "cluster", target="A",
                          reference="siblings", device="cpu")
    return res, res.top("A", k)

res1, p1 = panel_from_random_split(1)
res2, p2 = panel_from_random_split(2)

print("split 1 ->", p1)
print("split 2 ->", p2)
print("genes in common:", len(set(p1) & set(p2)), "of 10")

split 1 -> ['G123', 'G096', 'G005', 'G016', 'G126', 'G059', 'G072', 'G018', 'G019', 'G041']
split 2 -> ['G103', 'G108', 'G068', 'G049', 'G031', 'G064', 'G012', 'G038', 'G042', 'G169']
genes in common: 0 of 10


/tmp/ipykernel_1559585/4062434936.py:6: ContrastQCWarning: [A] contrast direction is UNRELIABLE (half-split cos_u = 0.06 < 0.9): target and reference do not separate enough in the representation to pin the direction, so the ranked genes may not reflect this distinction. Consider a cleaner/larger reference, more cells, or a different encoder.
  res = recast.attribute(StubEncoder(NG), A, "cluster", target="A",
/tmp/ipykernel_1559585/4062434936.py:6: ContrastQCWarning: [A] contrast direction is UNRELIABLE (half-split cos_u = -0.05 < 0.9): target and reference do not separate enough in the representation to pin the direction, so the ranked genes may not reflect this distinction. Consider a cleaner/larger reference, more cells, or a different encoder.
  res = recast.attribute(StubEncoder(NG), A, "cluster", target="A",


Two confident-looking ten-gene panels for the same non-existent cluster, sharing nothing. Neither
run raised an exception, and nothing about the shape of either list says "this is noise". If you
had run only one of them and put it in a figure, there would have been no moment at which the
method told you it had failed — except the one it *did* print above.

## What the QC said

Both calls emitted a `ContrastQCWarning` (visible in the output of the previous cell, because
`qc="warn"` is the default). The numbers behind it:

In [3]:
res1.qc.round(3)

,n_target,n_reference,dprime,cos_u_mean,cos_u_min
A,200.0,200.0,1.535,0.063,0.021


Look carefully at which column caught it.

- **`cos_u_mean` = 0.06.** The cells on each side are split in half at random, `u` is
  re-estimated on each half, and the two estimates are compared. A reproducible direction scores
  near 1. This one scores near zero — re-estimating on different cells points somewhere else
  entirely, which is precisely why the two panels above disagree.
- **`dprime` ≈ 1.5 — and that looks fine.** It is above the `d' < 0.5` warning threshold, and a
  reader eyeballing it would call this a separated contrast.

`d'` is measured along the very direction that was fitted to these cells, so noise fitted
in-sample projects as apparent separation. It is a real quantity, but it cannot detect *this*
failure. `cos_u` can, because half-splitting is the only part of the QC that estimates the
direction on one set of cells and judges it on another.

**The two diagnostics are not redundant, and `cos_u` is the one that catches an invented
boundary.**

## How much signal does it take?

The degenerate case is the extreme. Real questionable contrasts are graded, so it is worth
watching the diagnostics come to life. We implant a genuine difference in 20 of the 200 genes and
increase its size.

In [4]:
signal = genes[:20]
rows = []
for delta in [0, 1, 2, 4, 8, 16]:
    A = ad.AnnData(X.copy()); A.var_names = genes
    lab = np.array(["A"] * NC); lab[NC // 2:] = "B"
    A.X[NC // 2:, :20] += delta                       # a real difference, only in group B
    A.obs["cluster"] = pd.Categorical(lab)
    r = recast.attribute(StubEncoder(NG), A, "cluster", target="B", reference="siblings",
                        device="cpu", qc="silent")
    rows.append({"added counts": delta,
                 "dprime": r.qc.loc["B", "dprime"],
                 "cos_u": r.qc.loc["B", "cos_u_mean"],
                 "implanted genes in top-20": len(set(r.top("B", 20)) & set(signal))})

pd.DataFrame(rows).round(3).set_index("added counts")

,dprime,cos_u,implanted genes in top-20
added counts,,,
0,1.390,0.018,3
1,2.123,0.407,19
2,3.385,0.690,20
4,5.958,0.874,20
8,10.311,0.951,20
16,16.395,0.979,20


The bottom of that table is a healthy contrast and the top row is the degenerate one, with
`cos_u` climbing monotonically between them. Two things are worth taking from it, and the second
is the one people get wrong.

**The `cos_u` ≈ 0 row recovers 3 of 20 implanted genes.** Drawing a top-20 list from 200 genes of
which 20 are implanted has an expectation of 2, so 3 is chance. The panel is not "somewhat
right" — it is a lottery ticket.

**Recovery saturates long before `cos_u` does.** By the second row, `cos_u` is well under the 0.9
threshold and yet nearly every implanted gene is already recovered. That is not a flaw in the
threshold; it means the two things measure different claims. `cos_u` asks whether the *direction
vector* is reproducible — a property spread over all 200 coordinates. The top of a ranking can be
stable while the direction as a whole is not, because the noise is distributed over the genes that
are not being ranked highly.

:::{admonition} So what does a warning actually oblige you to do?
:class: tip

Not to throw the panel away. To stop treating the ranking as evidence about the target-versus-
reference distinction until you have checked one of these:

- **Is the boundary real?** The failure above was an invented cluster. Merging it back into its
  neighbour is the fix, and no gene list will tell you that.
- **Is the reference sane?** [Tutorial 3](03_choosing_the_reference) — a target and a reference
  that overlap biologically produce a weak contrast honestly.
- **Are there enough cells?** `cos_u` needs at least four per side to be estimable at all, and
  half-split estimates from a few dozen cells are noisy in their own right. A low `cos_u` on a
  20-cell cluster is a statement about the cluster's size, not necessarily about its biology.

And note the boundary of what QC covers: it is computed from embeddings alone, so it diagnoses
the *contrast*. It does not check your preprocessing — [Tutorial
2](02_subtype_markers) ends with a case where the QC is identical between a correct run and a
misnormalized one.
:::

## Where to go next

- [Tutorial 5](05_scoring_and_transfer) — take a panel you now trust, freeze it, and score it on
  cells that had no part in selecting it.